# Local Experiment Data Processing

Run the cells in order. Results stay in memory; the notebook does not write output files.

In [49]:
from datetime import time
from pathlib import Path

import numpy as np
import pandas as pd

DATA_RELATIVE_PATH = Path(
    "dataset/row/local_experiment_data/local_experiment_data_combined.csv"
)
INPUT_FILE = next(
    (
        root / DATA_RELATIVE_PATH
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / DATA_RELATIVE_PATH).is_file()
    ),
    None,
)
if INPUT_FILE is None:
    raise FileNotFoundError(f"Could not find {DATA_RELATIVE_PATH} from {Path.cwd()}")

data = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(data):,} observations from {INPUT_FILE}")

Loaded 27,468 observations from c:\FYP\coding\dataset\row\local_experiment_data\local_experiment_data_combined.csv


## Step 1: Add Malaysia Local Time

Convert the Unix `ts` value to Malaysia Time (UTC+8) and add it as `datetime_malaysia`. The original `ts` values remain unchanged.

In [50]:
TIMEZONE = "Asia/Kuala_Lumpur"

data["datetime_malaysia"] = pd.to_datetime(
    data["ts"], unit="s", utc=True
).dt.tz_convert(TIMEZONE)

print(f"Added Malaysia local time (UTC+8) for {len(data):,} observations.")

Added Malaysia local time (UTC+8) for 27,468 observations.


## Step 2: Time Filtering

Keep observations from 10:00:00 through 17:00:00, inclusive, using `datetime_malaysia`. Original timestamps and date information remain unchanged.

In [51]:
START_TIME = time(10, 0)
END_TIME = time(17, 0)

local_times = data["datetime_malaysia"].dt.time
time_filter = local_times.between(START_TIME, END_TIME, inclusive="both")
filtered_data = data.loc[time_filter].copy()

print(f"Input observations: {len(data):,}")
print(f"Observations from 10:00:00 to 17:00:00: {len(filtered_data):,}")

Input observations: 27,468
Observations from 10:00:00 to 17:00:00: 15,504


## Step 3: Rain Detection

A reading with at least 3 of the 4 LDR sensors at zero marks rain. Flag observations within 10 minutes before or after each trigger, inclusive. Keep all original row values, including the LDR readings.

In [52]:
LDR_COLUMNS = ["ldr1", "ldr2", "ldr3", "ldr4"]
RAIN_PADDING_SECONDS = 10 * 60

timestamp_seconds = pd.to_numeric(filtered_data["ts"], errors="raise").to_numpy(
    dtype=np.int64
)
rain_trigger = filtered_data[LDR_COLUMNS].eq(0).sum(axis=1).ge(3).to_numpy()
zero_timestamps = np.sort(timestamp_seconds[rain_trigger])

rain_detected = np.zeros(len(filtered_data), dtype=bool)
if zero_timestamps.size:
    positions = np.searchsorted(zero_timestamps, timestamp_seconds)
    nearest_zero_distance = np.full(
        len(filtered_data), np.iinfo(np.int64).max, dtype=np.int64
    )
    has_left_neighbor = positions > 0
    has_right_neighbor = positions < len(zero_timestamps)
    nearest_zero_distance[has_left_neighbor] = np.abs(
        timestamp_seconds[has_left_neighbor]
        - zero_timestamps[positions[has_left_neighbor] - 1]
    )
    nearest_zero_distance[has_right_neighbor] = np.minimum(
        nearest_zero_distance[has_right_neighbor],
        np.abs(
            timestamp_seconds[has_right_neighbor]
            - zero_timestamps[positions[has_right_neighbor]]
        ),
    )
    rain_detected = nearest_zero_distance <= RAIN_PADDING_SECONDS

rain_processed_data = filtered_data.copy()
rain_processed_data["rain_detected"] = rain_detected

print(f"Rain trigger readings (3 or more LDRs at zero): {int(rain_trigger.sum()):,}")
print(f"Rain-affected observations: {int(rain_detected.sum()):,}")
print(f"Rain-processed observations: {len(rain_processed_data):,}")

Rain trigger readings (3 or more LDRs at zero): 2,643
Rain-affected observations: 3,221
Rain-processed observations: 15,504


## Step 4: 10-Minute Resampling

Aggregate each calendar day separately into right-closed, non-overlapping 10-minute intervals labeled by each interval's end time. Numerical measurements use arithmetic means. `valid_observations` counts source rows in each interval, with 10 observations expected per interval, and `rain_detected_count` counts rain-flagged rows. Empty intervals and missing measurements remain NaN rather than being filled with zero.

In [53]:
RESAMPLE_FREQUENCY = "10min"
EXPECTED_OBSERVATIONS_PER_INTERVAL = 10

numeric_columns = [
    column
    for column in rain_processed_data.select_dtypes(include="number").columns
    if column != "ts"
]
resampled_days = []

for calendar_date, day_data in rain_processed_data.groupby(
    rain_processed_data["datetime_malaysia"].dt.date,
    sort=True,
):
    day_data = day_data.sort_values("datetime_malaysia").set_index(
        "datetime_malaysia"
    )
    resampler = day_data.resample(
        RESAMPLE_FREQUENCY,
        closed="right",
        label="right",
        origin="start_day",
    )
    interval_means = resampler[numeric_columns].mean()
    valid_counts = resampler.size()
    rain_counts = resampler["rain_detected"].sum(min_count=1)

    day_start = pd.Timestamp(calendar_date, tz=TIMEZONE)
    interval_ends = pd.date_range(
        start=day_start + pd.Timedelta(hours=10),
        end=day_start + pd.Timedelta(hours=17),
        freq=RESAMPLE_FREQUENCY,
    )
    interval_means = interval_means.reindex(interval_ends)
    valid_counts = valid_counts.reindex(interval_ends, fill_value=0).astype("int64")
    rain_counts = rain_counts.reindex(interval_ends).astype("Int64")

    interval_data = interval_means
    interval_data["valid_observations"] = valid_counts
    interval_data["rain_detected_count"] = rain_counts
    interval_data.insert(0, "datetime_malaysia", interval_ends)
    resampled_days.append(interval_data.reset_index(drop=True))

resampled_data = pd.concat(resampled_days, ignore_index=True)
print(f"10-minute intervals: {len(resampled_data):,}")
print(
"Intervals with at least 10 valid observations: "
f"{int(resampled_data['valid_observations'].ge(EXPECTED_OBSERVATIONS_PER_INTERVAL).sum()):,}"
)
print(
    "Intervals with no observations: "
    f"{int(resampled_data['valid_observations'].eq(0).sum()):,}"
)
print(
    "Rain-flagged observations counted in intervals: "
    f"{int(resampled_data['rain_detected_count'].sum(skipna=True)):,}"
)
rain_interval_counts = resampled_data["rain_detected_count"]
print(f"Rows in resampled dataset: {len(resampled_data):,}")
print(f"Intervals with rain (count > 0): {int(rain_interval_counts.gt(0).sum()):,}")
print(f"Intervals without rain (count = 0): {int(rain_interval_counts.eq(0).sum()):,}")

10-minute intervals: 1,634
Intervals with at least 10 valid observations: 1,491
Intervals with no observations: 53
Rain-flagged observations counted in intervals: 3,221
Rows in resampled dataset: 1,634
Intervals with rain (count > 0): 343
Intervals without rain (count = 0): 1,238


## Step 5: Missing Data and Quality Assessment

A configurable minimum valid-observation fraction determines whether an interval has enough data for a reliable assessment. Intervals below that minimum are flagged as raining as requested. For intervals meeting the minimum, flag rain when less than 80% of valid observations are not raining.

In [54]:
MIN_VALID_OBSERVATION_FRACTION = 0.80
MIN_NON_RAINING_FRACTION = 0.80

minimum_valid_observations = int(
    np.ceil(EXPECTED_OBSERVATIONS_PER_INTERVAL * MIN_VALID_OBSERVATION_FRACTION)
)
valid_observations = resampled_data["valid_observations"]
rain_observations = resampled_data["rain_detected_count"].fillna(0).astype("int64")
non_raining_observations = valid_observations - rain_observations

resampled_data["data_quality_pass"] = valid_observations.ge(
    minimum_valid_observations
)
resampled_data["non_raining_percentage"] = (
    non_raining_observations.div(valid_observations.where(valid_observations.gt(0)))
    * 100
)
below_minimum_valid = ~resampled_data["data_quality_pass"]
below_non_raining_threshold = resampled_data["non_raining_percentage"].lt(
    MIN_NON_RAINING_FRACTION * 100
)
resampled_data["rain_detected"] = below_minimum_valid | below_non_raining_threshold

print(
    "Minimum valid observations per interval: "
    f"{minimum_valid_observations} of {EXPECTED_OBSERVATIONS_PER_INTERVAL}"
)
print(
    "Intervals below minimum data quality: "
    f"{int(below_minimum_valid.sum()):,}"
)
print(
    "Intervals flagged as raining (including low-quality intervals): "
    f"{int(resampled_data['rain_detected'].sum()):,}"
)

Minimum valid observations per interval: 8 of 10
Intervals below minimum data quality: 88
Intervals flagged as raining (including low-quality intervals): 422


In [55]:
OUTPUT_FILE = INPUT_FILE.parents[3] / "dataset" / "processed" / "experiment_processed.csv"
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

resampled_data.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(resampled_data):,} rows to {OUTPUT_FILE}")

Saved 1,634 rows to c:\FYP\coding\dataset\processed\experiment_processed.csv
